# Predicting food product shelf life Using Machine Learning and Physicochemical, Microbial, Sensory, and Volatile Indicators

# ========================================
0. Objective
1. Import libraries
2. Load dataset
3. Clean column names
4. Dataset inspection
5. Define metadata, features, and target
6. Feature engineering
7. Exploratory data analysis
8. Prepare datasets for PCA/clustering
9. Scale datasets
10. PCA analysis
11. Clustering analysis
12. Classification modeling benchmark
13. Final train/test evaluation
14. Model interpretation
15. Export tables
# ========================================

## Objectives
1. Develop interpretable machine learning models to predict food product acceptability during storage.
2. Integrate physicochemical, microbial, sensory, volatile, and storage-condition indicators.
3. Identify key variables driving quality deterioration.
4. Compare model performance across multiple classification algorithms using rolling time-aware validation.
5. Interpret the best-performing models using feature importance, coefficients, and SHAP.
#
#### IMPORTANT VALIDATION NOTE:
#### This notebook uses rolling time-aware validation instead of random cross-validation.
#### Earlier storage days are used for training and the next future day is used for validation.
#### This gives a more realistic estimate of future predictive performance.

## 1. Import libraries

In [ ]:
# ==============================
# 1. Import libraries
# ==============================

# Data handling

import os
import warnings
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy.stats import pearsonr
from scipy.cluster.hierarchy import dendrogram, linkage
from mpl_toolkits.mplot3d import Axes3D

# Preprocessing and dimensionality reduction
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.base import clone
# Time-aware validation uses custom day-based folds


# Clustering
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

# Model selection
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate

# Classification models
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

# Pipeline
from sklearn.pipeline import Pipeline

# Hierarchical clustering visualization
from scipy.cluster.hierarchy import dendrogram, linkage

# SHAP
import shap

# Display settings
pd.set_option("display.max_columns", None)

## 2. Load dataset

## The dataset used in this project is not publicly available due to confidentiality.
### In practice, the data was loaded from an internal Excel file containing physicochemical, microbiological, and sensory measurements collected over storage time.

In [ ]:
# ==============================
# 2. Load dataset
# ==============================

# NOTE:
# The dataset is not included due to confidentiality.
# In the original analysis, data was loaded from an internal Excel file.

# Example (not executed here):
# df = pd.read_excel("path_to_dataset.xlsx")

# Placeholder to illustrate structure
df = pd.DataFrame()

## Dataset Description
### The dataset contains measurements collected during storage (0, 5, 10 °C) of food samples.
### Measurements include:
1. Physicochemical: pH, color (L*, a*, b*), texture parameters
2. Chemical spoilage indicators: TBARS, peroxide value, carbonyl content
3. Microbial: total plate count (TPC), lactic acid bacteria (LAB)
4. Sensory: Odor, Color
5. Volatile compounds
#
### Target column (Acceptability):
### 0 = Acceptable
### 1 = Rejected

## 3. Clean column names

In [ ]:
# ==============================
# 3. Clean column names
# ==============================

df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace("*", "", regex=False)
)

df = df.rename(columns={
    "ButandIine_2_3": "Butanedione_2_3",
    "Butanoz_2": "Butanol_2",
    "Odour_sensory": "Odor_sensory",
    "Colour_sensory": "Color_sensory"
})

# Create a group identifier for each experimental condition.
# This is useful for triplicate-aware checks and must not be used as a model feature.
if {"Day", "Temperature"}.issubset(df.columns):
    df["Group_ID"] = df["Day"].astype(str) + "_" + df["Temperature"].astype(str)


## 4. Dataset inspection

In [ ]:
# ==============================
# 4. Dataset inspection
# ==============================

display(df.head())
print("Shape:", df.shape)
df.info()
display(df.describe(include="all"))
display(df.isnull().sum().sort_values(ascending=False))

## 5. Define metadata, features, and target

In [ ]:
# ==============================
# 5. Define metadata, features, and target
# ==============================

id_cols = ["Sample_code", "Group_ID"]
storage_cols = ["Day", "Temperature"]
target_col = "Acceptability"

# Sort by time before any time-aware modeling
sort_cols = [col for col in ["Day", "Temperature", "Sample_code"] if col in df.columns]
df = df.sort_values(sort_cols).reset_index(drop=True)

# Use all predictor columns for EDA/modeling and PCA/clustering.
# Acceptability is the target and is excluded. Sample_code and Group_ID are identifiers and are excluded.
feature_cols_all = [col for col in df.columns if col not in id_cols + [target_col]]
feature_cols_pca_cluster = feature_cols_all.copy()

X_all = df[feature_cols_all].copy()
X_pca_cluster = df[feature_cols_pca_cluster].copy()
y = df[target_col].copy()

print("Number of features for EDA/modeling:", len(feature_cols_all))
print("Number of features for PCA/clustering:", len(feature_cols_pca_cluster))
print("Acceptability in X?", "Acceptability" in X_all.columns)
print("Sample_code in X?", "Sample_code" in X_all.columns)
print("Group_ID in X?", "Group_ID" in X_all.columns)
print("Target distribution:")
display(y.value_counts())
print("Day distribution:")
display(df["Day"].value_counts().sort_index())
print("Day x Temperature:")
display(pd.crosstab(df["Day"], df["Temperature"]))
print("Day x Acceptability:")
display(pd.crosstab(df["Day"], df[target_col]))


## 6. Feature engineering

In [ ]:
# ==============================
# 6. Feature engineering
# ==============================

if {"TBARS", "PV"}.issubset(df.columns):
    df["TBARS_to_PV"] = df["TBARS"] / (df["PV"] + 1e-6)

# Rebuild feature sets after adding engineered features.
# Day, Temperature, and TBARS_to_PV are included as predictors.
feature_cols_all = [col for col in df.columns if col not in id_cols + [target_col]]
feature_cols_pca_cluster = feature_cols_all.copy()

X_all = df[feature_cols_all].copy()
X_pca_cluster = df[feature_cols_pca_cluster].copy()
y = df[target_col].copy()

print("Updated number of features for EDA/modeling:", len(feature_cols_all))
print("Updated number of features for PCA/clustering:", len(feature_cols_pca_cluster))
print("Day included?", "Day" in X_pca_cluster.columns)
print("Temperature included?", "Temperature" in X_pca_cluster.columns)
print("TBARS_to_PV included?", "TBARS_to_PV" in X_pca_cluster.columns)
print("Acceptability in X?", "Acceptability" in X_all.columns)
print("Sample_code in X?", "Sample_code" in X_all.columns)
print("Group_ID in X?", "Group_ID" in X_all.columns)


## 7. Exploratory data analysis

### 7.1 Correlation heatmap

In [ ]:
# ==============================
# 7.1 Correlation heatmap
# ==============================

corr = X_all.corr(numeric_only=True)
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Feature Correlation Matrix")
plt.show()

### 7.2 Strongest feature-feature correlations

In [ ]:
# ==============================
# 7.2 Strongest feature-feature correlations
# ==============================

corr_pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
corr_pairs = corr_pairs.stack().reset_index()
corr_pairs.columns = ["Feature_1", "Feature_2", "Correlation"]
corr_pairs["Abs_Correlation"] = corr_pairs["Correlation"].abs()
top_corr = corr_pairs.sort_values("Abs_Correlation", ascending=False)
display(top_corr.head(20))

### 7.3 Correlation with target

In [ ]:
# ==============================
# 7.3 Correlation with target
# ==============================

numeric_for_target_corr = df[
    [col for col in feature_cols_all if pd.api.types.is_numeric_dtype(df[col])] + [target_col]
]

target_corr = numeric_for_target_corr.corr()[target_col].sort_values(ascending=False)
display(target_corr)

target_corr_abs = target_corr.drop(target_col).abs().sort_values(ascending=False)
display(target_corr_abs.head(25))

### 7.4 Annotated smaller correlation matrix with significance stars

In [ ]:
# ==============================
# 8.4 Annotated small correlation matrix
# ==============================

def p_to_star(p):
    if p <= 0.0001:
        return "****"
    elif p <= 0.001:
        return "***"
    elif p <= 0.01:
        return "**"
    elif p <= 0.05:
        return "*"
    else:
        return ""

corr_top_candidates = [
    col for col in target_corr_abs.index.tolist()
    if pd.api.types.is_numeric_dtype(df[col])
]
top_features_for_corr = corr_top_candidates[:12]
corr_small = df[top_features_for_corr].corr()
annot = corr_small.copy().astype(str)

for i, row_var in enumerate(top_features_for_corr):
    for j, col_var in enumerate(top_features_for_corr):
        r, p = pearsonr(df[row_var], df[col_var])
        annot.iloc[i, j] = f"{r:.2f} \n {p_to_star(p)}"

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_small,
    annot=annot,
    fmt="",
    cmap="coolwarm",
    center=0,
    square=True
)
plt.title("Annotated Correlation Matrix (Top Features)")
plt.show()

### 7.5 Class distribution

In [ ]:
# ==============================
# 7.4 Class distribution
# ==============================

plt.figure(figsize=(5, 4))
sns.countplot(x=target_col, data=df)
plt.title("Acceptability Class Distribution")
plt.xlabel("Acceptability")
plt.ylabel("Count")
plt.show()

### 7.6 Selected scatter plots

In [ ]:
# ==============================
# 7.5 Selected scatter plots
# ==============================

scatter_pairs = [
    ("TPC_log", "LAB_log"),
    ("TPC_log", "TVBN"),
    ("TVBN", "Butanone_2"),
    ("PV", "TBARS"),
    ("Odor_sensory", "TVBN"),
    ("pentanal", "Hexanal"),
    ("TBARS_to_PV", "Acceptability")
]

for x_var, y_var in scatter_pairs:
    if x_var in df.columns and y_var in df.columns:
        plt.figure(figsize=(6, 4))
        sns.scatterplot(data=df, x=x_var, y=y_var, hue="Acceptability", s=80)
        plt.title(f"{y_var} vs {x_var}")
        plt.show()

### 7.7 Boxplots by target class

In [ ]:
# ==============================
# 7.6 Boxplots by target class
# ==============================

boxplot_features = [
    "TPC_log", "LAB_log", "TVBN", "Odor_sensory", "Color_L",
    "Firmness", "Stickiness", "Hexanol", "TBARS_to_PV"
]

for col in boxplot_features:
    if col in df.columns:
        plt.figure(figsize=(5, 4))
        sns.boxplot(data=df, x="Acceptability", y=col)
        plt.title(f"{col} by Acceptability")
        plt.show()

### 7.8 Trends over storage day

In [ ]:
# ==============================
# 7.7 Trends over storage day
# ==============================

trend_features = [
    "TPC_log", "TVBN", "Odor_sensory", "Color_L",
    "Firmness", "Hexanal", "TBARS_to_PV"
]

for col in trend_features:
    if col in df.columns:
        plt.figure(figsize=(6, 4))
        sns.scatterplot(data=df, x="Day", y=col, hue="Temperature", style="Acceptability", s=90)
        plt.title(f"{col} vs Storage Day")
        plt.show()

### 7.9 Grouped distributions

In [ ]:
# ==============================
# 7.8 Grouped distributions (with KDE line)
# ==============================

grouped_features = {
    "Physicochemical": ["Color_L", "Color_a", "Color_b", "Firmness", "Work_of_shear", "Stickiness"],
    "Chemical": ["pH", "Carbonyl", "PV", "TBARS", "TVBN", "TBARS_to_PV"],
    "Microbiological": ["TPC_log", "LAB_log"],
    "Sensory": ["Odor_sensory", "Color_sensory"],
    "Volatile": [
        "Propanol_1", "Butanedione_2_3", "Butanone_2", "Butanol_2", "Acetic_acid",
        "pentanal", "Propanoic_acid_2_oxo", "Hexanal", "Hexanol", "Ethanol_2_butoxy",
        "Benzaldehyde", "Benzeneacetaldehyde"
    ]
}

for group_name, cols in grouped_features.items():
    available_cols = [c for c in cols if c in df.columns]
    if not available_cols:
        continue

    n_cols = 3
    n_rows = (len(available_cols) + n_cols - 1) // n_cols

    plt.figure(figsize=(14, 4 * n_rows))
    for i, col in enumerate(available_cols, 1):
        plt.subplot(n_rows, n_cols, i)
        sns.histplot(df[col], bins=15, kde=True)
        plt.title(col)
    plt.suptitle(f"{group_name} Feature Distributions", y=1.02)
    plt.tight_layout()
    plt.show()

## 8. Prepare datasets for PCA/clustering

In [ ]:
# ==============================
# 8. Prepare datasets for PCA/clustering
# ==============================

X_cluster_all = X_pca_cluster.copy()
X_cluster_no_micro = X_pca_cluster.drop(columns=[c for c in ["TPC_log", "LAB_log"] if c in X_pca_cluster.columns]).copy()

print("With microbial shape:", X_cluster_all.shape)
print("Without microbial shape:", X_cluster_no_micro.shape)

## 9. Scale datasets

In [ ]:
# ==============================
# 9. Scale datasets
# ==============================

# PCA and clustering are exploratory visualizations in this notebook.
# Therefore, scaling is fitted on the full exploratory dataset only for visualization.
# Model evaluation later uses pipelines so scalers are fitted only on training data within each fold/split.

scaler_all = StandardScaler()
X_cluster_all_scaled = pd.DataFrame(
    scaler_all.fit_transform(X_cluster_all),
    columns=X_cluster_all.columns,
    index=X_cluster_all.index
)

scaler_no_micro = StandardScaler()
X_cluster_no_micro_scaled = pd.DataFrame(
    scaler_no_micro.fit_transform(X_cluster_no_micro),
    columns=X_cluster_no_micro.columns,
    index=X_cluster_no_micro.index
)


In [ ]:
print("Shape:", X_cluster_all.shape)
print("Columns:", X_cluster_all.columns.tolist())

print("No micro shape:", X_cluster_no_micro.shape)
print("No micro columns:", X_cluster_no_micro.columns.tolist())

## 10. PCA analysis

### 10.1 PCA — with microbial

In [ ]:
# ==============================
# 10.1 PCA — WITH microbial
# ==============================

pca_all = PCA(n_components=3, random_state=42)
X_pca_all = pca_all.fit_transform(X_cluster_all_scaled)

df_pca_all = pd.DataFrame(X_pca_all, columns=["PC1", "PC2", "PC3"], index=df.index)
df_pca_all["Acceptability"] = df["Acceptability"]
df_pca_all["Day"] = df["Day"]
df_pca_all["Temperature"] = df["Temperature"]

loadings_all = pd.DataFrame(
    pca_all.components_.T,
    columns=["PC1", "PC2", "PC3"],
    index=X_cluster_all.columns
)

print("Explained variance ratio:", pca_all.explained_variance_ratio_)
print("Total variance explained (3 PCs):", pca_all.explained_variance_ratio_.sum())

### 10.2 PCA — without microbial

In [ ]:
# ==============================
# 10.2 PCA — WITHOUT microbial
# ==============================

pca_no_micro = PCA(n_components=3, random_state=42)
X_pca_no_micro = pca_no_micro.fit_transform(X_cluster_no_micro_scaled)

df_pca_no_micro = pd.DataFrame(X_pca_no_micro, columns=["PC1", "PC2", "PC3"], index=df.index)
df_pca_no_micro["Acceptability"] = df["Acceptability"]
df_pca_no_micro["Day"] = df["Day"]
df_pca_no_micro["Temperature"] = df["Temperature"]

loadings_no_micro = pd.DataFrame(
    pca_no_micro.components_.T,
    columns=["PC1", "PC2", "PC3"],
    index=X_cluster_no_micro.columns
)

print("Explained variance ratio:", pca_no_micro.explained_variance_ratio_)
print("Total variance explained (3 PCs):", pca_no_micro.explained_variance_ratio_.sum())

### 10.3 2D PCA plots with percentages

In [ ]:
# ==============================
# 10.3 PCA 2D plots
# ==============================

plt.figure(figsize=(7, 5))
sns.scatterplot(data=df_pca_all, x="PC1", y="PC2", hue="Acceptability", palette="coolwarm", s=100)
plt.xlabel(f"PC1 ({pca_all.explained_variance_ratio_[0] * 100:.2f}%)")
plt.ylabel(f"PC2 ({pca_all.explained_variance_ratio_[1] * 100:.2f}%)")
plt.title("PCA Projection (WITH Microbial)")
plt.axhline(0, linestyle="--", color="gray")
plt.axvline(0, linestyle="--", color="gray")
plt.show()

plt.figure(figsize=(7, 5))
sns.scatterplot(data=df_pca_no_micro, x="PC1", y="PC2", hue="Acceptability", palette="coolwarm", s=100)
plt.xlabel(f"PC1 ({pca_no_micro.explained_variance_ratio_[0] * 100:.2f}%)")
plt.ylabel(f"PC2 ({pca_no_micro.explained_variance_ratio_[1] * 100:.2f}%)")
plt.title("PCA Projection (WITHOUT Microbial)")
plt.axhline(0, linestyle="--", color="gray")
plt.axvline(0, linestyle="--", color="gray")
plt.show()

### 10.4 3D PCA plots

In [ ]:
# ==============================
# 10.4 PCA 3D plots 
# ==============================
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(
    df_pca_all["PC1"], df_pca_all["PC2"], df_pca_all["PC3"],
    c=df_pca_all["Acceptability"], s=50
)
ax.set_xlabel(f"PC1 ({pca_all.explained_variance_ratio_[0] * 100:.2f}%)")
ax.set_ylabel(f"PC2 ({pca_all.explained_variance_ratio_[1] * 100:.2f}%)")
ax.set_zlabel(f"PC3 ({pca_all.explained_variance_ratio_[2] * 100:.2f}%)")
ax.set_title("3D PCA (WITH Microbial)")
plt.show()

### 10.5 PCA comparison

In [ ]:
# ==============================
# 10.5 PCA comparison
# ==============================

comparison_pca = pd.DataFrame({
    "PC1_variance": [pca_all.explained_variance_ratio_[0], pca_no_micro.explained_variance_ratio_[0]],
    "PC2_variance": [pca_all.explained_variance_ratio_[1], pca_no_micro.explained_variance_ratio_[1]],
    "Total_variance": [pca_all.explained_variance_ratio_.sum(), pca_no_micro.explained_variance_ratio_.sum()]
}, index=["With_microbial", "Without_microbial"])

display(comparison_pca)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.scatterplot(data=df_pca_all, x="PC1", y="PC2", hue="Acceptability", palette="coolwarm", s=100, ax=axes[0])
axes[0].set_title("WITH Microbial Features")
sns.scatterplot(data=df_pca_no_micro, x="PC1", y="PC2", hue="Acceptability", palette="coolwarm", s=100, ax=axes[1])
axes[1].set_title("WITHOUT Microbial Features")
for ax in axes:
    ax.axhline(0, linestyle="--", color="gray")
    ax.axvline(0, linestyle="--", color="gray")
plt.tight_layout()
plt.show()

top_with = loadings_all["PC1"].abs().sort_values(ascending=False).head(10)
top_without = loadings_no_micro["PC1"].abs().sort_values(ascending=False).head(10)
comparison_loadings = pd.DataFrame({
    "With_microbial": top_with.index,
    "Without_microbial": top_without.index
})
display(comparison_loadings)

## 11. Clustering analysis

### 11.1 KMeans k=2 — with microbial

In [ ]:
# ==============================
# 11.1 KMeans k=2 — WITH microbial
# ==============================

kmeans_all = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters_all = kmeans_all.fit_predict(X_cluster_all_scaled)
df_pca_all["Cluster_k2"] = clusters_all

display(pd.crosstab(df_pca_all["Cluster_k2"], df_pca_all["Acceptability"]))

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df_pca_all, x="PC1", y="PC2", hue="Cluster_k2", palette="Set1", s=100)
plt.title("KMeans (k=2) WITH Microbial")
plt.axhline(0, linestyle="--", color="gray")
plt.axvline(0, linestyle="--", color="gray")
plt.show()

### 11.2 KMeans k=2 — without microbial

In [ ]:
# ==============================
# 11.2 KMeans k=2 — WITHOUT microbial
# ==============================

kmeans_no_micro = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters_no_micro = kmeans_no_micro.fit_predict(X_cluster_no_micro_scaled)
df_pca_no_micro["Cluster_k2"] = clusters_no_micro

display(pd.crosstab(df_pca_no_micro["Cluster_k2"], df_pca_no_micro["Acceptability"]))

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df_pca_no_micro, x="PC1", y="PC2", hue="Cluster_k2", palette="Set1", s=100)
plt.title("KMeans (k=2) WITHOUT Microbial")
plt.axhline(0, linestyle="--", color="gray")
plt.axvline(0, linestyle="--", color="gray")
plt.show()

### 11.3 KMeans k=3 — with microbial

In [ ]:
# ==============================
# 11.3 KMeans k=3 — WITH microbial
# ==============================

kmeans_all_3 = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters_all_3 = kmeans_all_3.fit_predict(X_cluster_all_scaled)
df_pca_all["Cluster_k3"] = clusters_all_3
df["Cluster_3_with_micro"] = clusters_all_3

display(pd.crosstab(pd.Series(clusters_all_3, name="Cluster"), df["Acceptability"]))

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df_pca_all, x="PC1", y="PC2", hue="Cluster_k3", palette="Set2", s=100)
plt.title("KMeans (k=3) WITH Microbial")
plt.axhline(0, linestyle="--", color="gray")
plt.axvline(0, linestyle="--", color="gray")
plt.show()

### 11.4 KMeans k=3 — without microbial

In [ ]:
# ==============================
# 11.4 KMeans k=3 — WITHOUT microbial
# ==============================

kmeans_no_micro_3 = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters_no_micro_3 = kmeans_no_micro_3.fit_predict(X_cluster_no_micro_scaled)
df_pca_no_micro["Cluster_k3"] = clusters_no_micro_3
df["Cluster_3_no_micro"] = clusters_no_micro_3

display(pd.crosstab(pd.Series(clusters_no_micro_3, name="Cluster"), df["Acceptability"]))

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df_pca_no_micro, x="PC1", y="PC2", hue="Cluster_k3", palette="Set2", s=100)
plt.title("KMeans (k=3) WITHOUT Microbial")
plt.axhline(0, linestyle="--", color="gray")
plt.axvline(0, linestyle="--", color="gray")
plt.show()

### 11.5 Hierarchical clustering — with microbial

In [ ]:
# ==============================
# 11.5 Hierarchical clustering — WITH microbial
# ==============================

agg_all = AgglomerativeClustering(n_clusters=2)
clusters_hier_all = agg_all.fit_predict(X_cluster_all_scaled)
df_pca_all["Cluster_hier"] = clusters_hier_all

display(pd.crosstab(pd.Series(clusters_hier_all, name="Cluster"), df["Acceptability"]))

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df_pca_all, x="PC1", y="PC2", hue="Cluster_hier", palette="Set1", s=100)
plt.title("Hierarchical Clustering WITH Microbial")
plt.axhline(0, linestyle="--", color="gray")
plt.axvline(0, linestyle="--", color="gray")
plt.show()

### 11.6 Hierarchical clustering — without microbial

In [ ]:
# ==============================
# 11.6 Hierarchical clustering — WITHOUT microbial
# ==============================

agg_no_micro = AgglomerativeClustering(n_clusters=2)
clusters_hier_no_micro = agg_no_micro.fit_predict(X_cluster_no_micro_scaled)
df_pca_no_micro["Cluster_hier"] = clusters_hier_no_micro

display(pd.crosstab(pd.Series(clusters_hier_no_micro, name="Cluster"), df["Acceptability"]))

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df_pca_no_micro, x="PC1", y="PC2", hue="Cluster_hier", palette="Set1", s=100)
plt.title("Hierarchical Clustering WITHOUT Microbial")
plt.axhline(0, linestyle="--", color="gray")
plt.axvline(0, linestyle="--", color="gray")
plt.show()

### 11.7 Dendrogram — with microbial

In [ ]:
# ==============================
# 11.7 Dendrogram — WITH microbial
# ==============================

linked = linkage(X_cluster_all_scaled, method="ward")
plt.figure(figsize=(10, 6))
dendrogram(linked)
plt.title("Dendrogram (WITH Microbial)")
plt.show()

### 11.8 Silhouette comparison

In [ ]:
# ==============================
# 11.8 Silhouette comparison
# ==============================

sil_summary = pd.DataFrame({
    "Model": [
        "KMeans k=2 (with micro)",
        "KMeans k=2 (no micro)",
        "KMeans k=3 (with micro)",
        "KMeans k=3 (no micro)",
        "Hierarchical (with micro)",
        "Hierarchical (no micro)"
    ],
    "Silhouette Score": [
        silhouette_score(X_cluster_all_scaled, clusters_all),
        silhouette_score(X_cluster_no_micro_scaled, clusters_no_micro),
        silhouette_score(X_cluster_all_scaled, clusters_all_3),
        silhouette_score(X_cluster_no_micro_scaled, clusters_no_micro_3),
        silhouette_score(X_cluster_all_scaled, clusters_hier_all),
        silhouette_score(X_cluster_no_micro_scaled, clusters_hier_no_micro)
    ]
})
display(sil_summary)

### 11.9 Cluster interpretation

In [ ]:
# ==============================
# 11.9 Cluster interpretation
# ==============================

important_features = [
    c for c in [
        "pH", "Color_L", "Color_a", "Color_b", "Firmness", "Work_of_shear", "Stickiness",
        "Carbonyl", "PV", "TBARS", "TVBN", "TPC_log", "LAB_log",
        "Odor_sensory", "Color_sensory", "Acceptability"
    ] if c in df.columns
]

display(df.groupby("Cluster_3_with_micro")[important_features].mean(numeric_only=True))
display(df.groupby("Cluster_3_no_micro")[important_features].mean(numeric_only=True))
display(pd.crosstab(df["Day"], df["Cluster_3_with_micro"]))
display(pd.crosstab(df["Day"], df["Cluster_3_no_micro"]))

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x="Cluster_3_with_micro", y="TPC_log", data=df)
plt.title("TPC_log by 3-Cluster Group (With Microbial)")
plt.show()

plt.figure(figsize=(6,4))
sns.boxplot(x="Cluster_3_with_micro", y="TVBN", data=df)
plt.title("TVBN by 3-Cluster Group (With Microbial)")
plt.show()

## 12. Classification modeling benchmark

### 12.1 Define target and feature sets

In [ ]:
# ==============================
# 12.1 Define target and feature sets
# ==============================

cluster_cols = [col for col in df.columns if "Cluster" in col]
model_id_cols = ["Sample_code", "Group_ID"]
y_model = df["Acceptability"].copy()

# Group_ID is used only as an identifier/check for triplicates and is not used as a model feature.
X_model_all = df.drop(columns=model_id_cols + ["Acceptability"] + cluster_cols, errors="ignore")
X_model_no_micro = X_model_all.drop(columns=[c for c in ["TPC_log", "LAB_log"] if c in X_model_all.columns])

print("Target distribution:")
print(y_model.value_counts())
print("Shape with microbial + storage:", X_model_all.shape)
print("Shape without microbial + storage:", X_model_no_micro.shape)
print("Acceptability in X?", "Acceptability" in X_model_all.columns)
print("Sample_code in X?", "Sample_code" in X_model_all.columns)
print("Group_ID in X?", "Group_ID" in X_model_all.columns)
print("Cluster columns removed:", cluster_cols)


### 12.2 Cross-validation setup

In [ ]:
# ==============================
# 12.2 Cross-validation setup
# ==============================

# Based on the actual class distribution in this dataset:
# - Day 0 and 3 contain only class 0
# - Day 7, 9, and 13 contain both classes
# - Day 15 and later contain only class 1
#
# Therefore, the meaningful rolling time-aware validation folds are:
# Fold 1: train on 0,3,7 -> validate on 9
# Fold 2: train on 0,3,7,9 -> validate on 13
#
# These folds respect time order and keep both classes in the validation set.

rolling_folds = [
    ([0, 3, 7], [9]),
    ([0, 3, 7, 9], [13])
]

print("Rolling folds used:")
for i, (train_days, val_days) in enumerate(rolling_folds, 1):
    print(f"Fold {i}: train days {train_days} -> validate day {val_days}")

### 12.3 Models

In [ ]:
# ==============================
# 12.3 Benchmark models
# ==============================


models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Ridge Classifier": Pipeline([
        ("scaler", StandardScaler()),
        ("model", RidgeClassifier())
    ]),
    "SGD Classifier": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SGDClassifier(random_state=42))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(probability=True, random_state=42))
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ]),
    "Nearest Centroid": Pipeline([
        ("scaler", StandardScaler()),
        ("model", NearestCentroid())
    ]),
    "Linear Discriminant Analysis": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearDiscriminantAnalysis())
    ]),
    "Gaussian NB": GaussianNB(),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
        eval_metric="logloss"
    )
}

### 12.4 Rolling time_aware evaluation function

In [ ]:
# ==============================
# 12.4 Rolling time-aware evaluation function
# ==============================

def evaluate_models_rolling_time(X, y, df_ref, models, rolling_folds):
    summary_rows = []
    fold_rows = []

    for model_name, model in models.items():
        model_fold_results = []

        for fold_idx, (train_days, val_days) in enumerate(rolling_folds, start=1):
            train_mask = df_ref["Day"].isin(train_days)
            val_mask = df_ref["Day"].isin(val_days)

            X_train = X.loc[train_mask]
            y_train = y.loc[train_mask]
            X_val = X.loc[val_mask]
            y_val = y.loc[val_mask]

            model_copy = clone(model)
            model_copy.fit(X_train, y_train)
            y_pred = model_copy.predict(X_val)

            fold_result = {
                "Model": model_name,
                "Fold": fold_idx,
                "Train_Days": str(train_days),
                "Validation_Days": str(val_days),
                "Accuracy": accuracy_score(y_val, y_pred),
                "Precision": precision_score(y_val, y_pred, zero_division=0),
                "Recall": recall_score(y_val, y_pred, zero_division=0),
                "F1 Score": f1_score(y_val, y_pred, zero_division=0)
            }

            fold_rows.append(fold_result)
            model_fold_results.append(fold_result)

        fold_df = pd.DataFrame(model_fold_results)
        summary_rows.append({
            "Model": model_name,
            "Accuracy": fold_df["Accuracy"].mean(),
            "Precision": fold_df["Precision"].mean(),
            "Recall": fold_df["Recall"].mean(),
            "F1 Score": fold_df["F1 Score"].mean(),
            "Num_Folds": len(fold_df)
        })

    results_df = pd.DataFrame(summary_rows).sort_values(by="F1 Score", ascending=False)
    fold_results_df = pd.DataFrame(fold_rows)
    return results_df, fold_results_df



### 12.5 Run models — with microbial

In [ ]:
# ==============================
# 12.5 Run models — WITH microbial
# ==============================

results_with_micro, fold_results_with_micro = evaluate_models_rolling_time(
    X_model_all, y_model, df, models, rolling_folds
)
display(results_with_micro)
display(fold_results_with_micro)

### 12.6 Run models — without microbial

In [ ]:
# ==============================
# 12.6 Run models — WITHOUT microbial
# ==============================

results_no_micro, fold_results_no_micro = evaluate_models_rolling_time(
    X_model_no_micro, y_model, df, models, rolling_folds
)
display(results_no_micro)
display(fold_results_no_micro)

### 12.7 Benchmark plots

In [ ]:
# ==============================
# 12.7 Benchmark plots
# ==============================

plt.figure(figsize=(10, 6))
sns.barplot(data=results_with_micro, y="Model", x="F1 Score")
plt.title("Rolling Time-Aware Model Benchmark (WITH Microbial)")
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(data=results_no_micro, y="Model", x="F1 Score")
plt.title("Rolling Time-Aware Model Benchmark (WITHOUT Microbial)")
plt.show()

## 13. Final train/test evaluation

### 13.1 Select final 4 models

## Model selection rationale:
### The final models selected for evaluation differ slightly between the random and time-aware validation workflows. In the random validation notebook, model selection was based primarily on predictive performance. In the time-aware notebook, greater emphasis was placed on stability across temporal folds and realistic future prediction performance. As a result, Ridge Classifier was included in the time-aware analysis because it showed more consistent behavior under temporal validation, whereas XGBoost showed greater variability.

In [ ]:
# ==============================
# 13.1 Final selected models
# ==============================

selected_models = {
    "Logistic Regression": models["Logistic Regression"],
    "Ridge Classifier": models["Ridge Classifier"],
    "Gradient Boosting": models["Gradient Boosting"],
    "Random Forest": models["Random Forest"]
}

### 13.2 Train/test split

## Note on final evaluation:
### Due to the limited dataset size, the storage days used in the final test set (Days 9 and 13) were also used during the rolling time-aware validation phase for model selection. Therefore, the final evaluation should be interpreted as a comparative assessment of the selected models rather than a fully independent holdout test. Future studies with larger datasets should reserve an external test set that is not used during model development.

In [ ]:
# ==============================
# 13.2 Train/test split — WITH microbial
# ==============================

train_days_final = [0, 3, 7]
test_days_final = [9, 13]

train_mask_final = df["Day"].isin(train_days_final)
test_mask_final = df["Day"].isin(test_days_final)

X_train_with = X_model_all.loc[train_mask_final].copy()
X_test_with = X_model_all.loc[test_mask_final].copy()
y_train_with = y_model.loc[train_mask_final].copy()
y_test_with = y_model.loc[test_mask_final].copy()

X_train_no = X_model_no_micro.loc[train_mask_final].copy()
X_test_no = X_model_no_micro.loc[test_mask_final].copy()
y_train_no = y_model.loc[train_mask_final].copy()
y_test_no = y_model.loc[test_mask_final].copy()

print("Final train days:", train_days_final)
print("Final test days:", test_days_final)
print("WITH microbial: train shape", X_train_with.shape, "| test shape", X_test_with.shape)
print("WITHOUT microbial: train shape", X_train_no.shape, "| test shape", X_test_no.shape)
print("Train target distribution:")
display(y_train_with.value_counts())
print("Test target distribution:")
display(y_test_with.value_counts())
print("Test set day x acceptability:")
display(pd.crosstab(df.loc[test_mask_final, "Day"], df.loc[test_mask_final, "Acceptability"]))

### 13.3 Train selected models

In [ ]:
# ==============================
# 13.3 Train selected models
# ==============================

trained_models_with = {}
trained_models_no = {}

for name, model in selected_models.items():
    model_copy_with = clone(model)
    model_copy_with.fit(X_train_with, y_train_with)
    trained_models_with[name] = model_copy_with

    model_copy_no = clone(model)
    model_copy_no.fit(X_train_no, y_train_no)
    trained_models_no[name] = model_copy_no

### 13.4 Test results

In [ ]:
# ==================================
# 13.4 Test results — WITH microbial
# ==================================

def evaluate_holdout_models(trained_models, X_test, y_test):
    rows = []
    for name, model in trained_models.items():
        y_pred = model.predict(X_test)
        rows.append({
            "Model": name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0)
        })
    return pd.DataFrame(rows).sort_values(by="F1 Score", ascending=False)

test_results_with_df = evaluate_holdout_models(trained_models_with, X_test_with, y_test_with)
test_results_no_df = evaluate_holdout_models(trained_models_no, X_test_no, y_test_no)

display(test_results_with_df)
display(test_results_no_df)


### 13.5 Classification reports

In [ ]:
# ============================================
# 13.5 Classification reports — WITH microbial
# ============================================

for name, model in trained_models_with.items():
    y_pred = model.predict(X_test_with)
    print(f"{name} — WITH microbial")
    print(classification_report(y_test_with, y_pred, zero_division=0))

In [ ]:
# ===============================================
# 13.5 Classification reports — WITHOUT microbial
# ===============================================

for name, model in trained_models_no.items():
    y_pred = model.predict(X_test_no)
    print(f"{name} — WITHOUT microbial")
    print(classification_report(y_test_no, y_pred, zero_division=0))

### 13.6 Confusion matrices

In [ ]:
# ========================================
# 13.6 Confusion matrices — WITH microbial
# ========================================

for name, model in trained_models_with.items():
    y_pred = model.predict(X_test_with)
    ConfusionMatrixDisplay.from_predictions(y_test_with, y_pred)
    plt.title(f"{name} — WITH microbial (test days 9 and 13)")
    plt.show()

In [ ]:
# ===========================================
# 13.6 Confusion matrices — WITHOUT microbial
# ===========================================

for name, model in trained_models_no.items():
    y_pred = model.predict(X_test_no)
    ConfusionMatrixDisplay.from_predictions(y_test_no, y_pred)
    plt.title(f"{name} — WITHOUT microbial (test days 9 and 13)")
    plt.show()

## 14. Model interpretation

### 14.1 Feature importance

#### 14.1.1 Gradient Boosting feature importance

In [ ]:
# =========================================
# 14.1.1 Feature importance - Gradient Boosting
# =========================================

# Use Gradient Boosting for tree-based importance in the final selected models.
gb_with = trained_models_with["Gradient Boosting"]
gb_no = trained_models_no["Gradient Boosting"]

gb_importance_with = pd.Series(gb_with.feature_importances_, index=X_train_with.columns).sort_values(ascending=False)
gb_importance_no = pd.Series(gb_no.feature_importances_, index=X_train_no.columns).sort_values(ascending=False)

display(gb_importance_with.head(15))
display(gb_importance_no.head(15))

plt.figure(figsize=(8, 6))
gb_importance_with.head(15).sort_values().plot(kind="barh")
plt.title("Gradient Boosting Feature Importance - WITH microbial")
plt.xlabel("Importance")
plt.show()

plt.figure(figsize=(8, 6))
gb_importance_no.head(15).sort_values().plot(kind="barh")
plt.title("Gradient Boosting Feature Importance - WITHOUT microbial")
plt.xlabel("Importance")
plt.show()

In [ ]:
# =========================================
# 14.1.2 Feature importance - Random Forest
# =========================================

# Use Random Forest for tree-based importance in the final selected models.
rf_with = trained_models_with["Random Forest"]
rf_no = trained_models_no["Random Forest"]

rf_importance_with = pd.Series(rf_with.feature_importances_, index=X_train_with.columns).sort_values(ascending=False)
rf_importance_no = pd.Series(rf_no.feature_importances_, index=X_train_no.columns).sort_values(ascending=False)

display(rf_importance_with.head(15))
display(rf_importance_no.head(15))

plt.figure(figsize=(8, 6))
rf_importance_with.head(15).sort_values().plot(kind="barh")
plt.title("Random Forest Feature Importance - WITH microbial")
plt.xlabel("Importance")
plt.show()

plt.figure(figsize=(8, 6))
rf_importance_no.head(15).sort_values().plot(kind="barh")
plt.title("Random Forest Feature Importance - WITHOUT microbial")
plt.xlabel("Importance")
plt.show()


### 14.2 Logistic Regression coefficients

In [ ]:
# ======================================
# 14.2.1 Logistic Regression coefficients
# ======================================

logreg_with = trained_models_with["Logistic Regression"]
logreg_no = trained_models_no["Logistic Regression"]

coef_with = pd.Series(
    logreg_with.named_steps["model"].coef_[0],
    index=X_train_with.columns
).sort_values(key=np.abs, ascending=False)

coef_no = pd.Series(
    logreg_no.named_steps["model"].coef_[0],
    index=X_train_no.columns
).sort_values(key=np.abs, ascending=False)

display(coef_with.head(20))
display(coef_no.head(20))

plt.figure(figsize=(8, 6))
coef_with.head(25).sort_values().plot(kind="barh")
plt.title("Logistic Regression Coefficients - WITH microbial")
plt.xlabel("Coefficient")
plt.show()

plt.figure(figsize=(8, 6))
coef_no.head(25).sort_values().plot(kind="barh")
plt.title("Logistic Regression Coefficients - WITHOUT microbial")
plt.xlabel("Coefficient")
plt.show()

### 14.3 Optional SHAP analysis

SHAP analysis was explored during development but is not executed in this GitHub-ready notebook to keep the workflow concise and fully reproducible for a general audience. The main interpretation is based on feature importance and logistic regression coefficients.

### 14.4 Compare important variables across interpretation methods

In [ ]:
# ===========================================
# 14.4 Compare top variables across interpretation methods
# ===========================================

comparison_with = pd.DataFrame({
    "LogReg_coef": pd.Series(coef_with.head(10).index),
    "GB_importance": pd.Series(gb_importance_with.head(10).index),
    "RF_importance": pd.Series(rf_importance_with.head(10).index)
})
display(comparison_with)

comparison_no = pd.DataFrame({
    "LogReg_coef": pd.Series(coef_no.head(10).index),
    "GB_importance": pd.Series(gb_importance_no.head(10).index),
    "RF_importance": pd.Series(rf_importance_no.head(10).index)
})
display(comparison_no)


### 14.5 Final comparison tables

#### 14.5.1 Benchmark summary

In [ ]:
# ==============================
# 14.5.1 Final benchmark summary
# ==============================

final_benchmark_summary = pd.DataFrame({
    "WITH_micro_rolling_F1": results_with_micro.set_index("Model")["F1 Score"],
    "WITHOUT_micro_rolling_F1": results_no_micro.set_index("Model")["F1 Score"]
})
display(final_benchmark_summary.sort_values(by="WITH_micro_rolling_F1", ascending=False))

final_test_summary = pd.DataFrame({
    "WITH_micro_test_9_13_F1": test_results_with_df.set_index("Model")["F1 Score"],
    "WITHOUT_micro_test_9_13_F1": test_results_no_df.set_index("Model")["F1 Score"]
})
display(final_test_summary.sort_values(by="WITH_micro_test_9_13_F1", ascending=False))

# ================================================================
# Final interpretation notes
# ================================================================
1. Main model selection is based on rolling time-aware validation.
2. Final testing uses days 9 and 13 because they are future relative to the training period and still contain both classes.
3. Days 15 and later were not used as the main test set because they contain only rejected samples.
4. This evaluation provides a more meaningful confusion matrix and a clearer comparison of model discrimination ability.
